# 📈 Stock Prediction System
## Enhanced ML Pipeline — Multiple Linear Regression + Gradient Boosting Ensemble

---

This notebook extracts the **core Machine Learning prediction logic** from the Django-based [Stock-Prediction-System-Application](https://github.com/masumganvir/stock_prediction_system) and enhances it with rich feature engineering and multiple models.

### 🔍 Original Source Files Used
| File | Function | Purpose |
|------|----------|---------|
| `app/views.py` | `predict(request, ticker_value, number_of_days)` | Core ML: data download, preprocessing, LinearRegression, future prediction |
| `app/views.py` | `index(request)` | yfinance download pattern |

### 🧠 Original ML Approach Faithfully Reproduced + Enhanced
- **Data**: `yf.download()` — same as `predict()` in views.py
- **Target**: Future price via `.shift(-forecast_out)` — views.py line 175
- **Scaling**: `preprocessing.scale(X)` — views.py line 178
- **Model 1**: `LinearRegression()` — views.py line 185 (preserved exactly)
- **Model 2+**: GradientBoosting, RandomForest, Ridge, VotingEnsemble
- **Split**: `train_test_split(..., test_size=0.2)` — views.py line 183

### 📌 Supported Tickers
- **US Stocks**: `AAPL`, `MSFT`, `TSLA`, `AMZN`, `GOOGL`, `NVDA`
- **Indian Stocks**: `RELIANCE.NS`, `TCS.NS`, `INFY.NS`, `HDFCBANK.NS`
- **Crypto**: `BTC-USD`, `ETH-USD`

---
## ⚙️ Section 1 — Install & Import Required Libraries

In [1]:
# Fix for Windows OpenBLAS memory allocation issue
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS']      = '1'
os.environ['MKL_NUM_THREADS']      = '1'
os.environ['NUMEXPR_NUM_THREADS']  = '1'
os.environ['MPLBACKEND']           = 'Agg'

import subprocess, sys
required = ['yfinance', 'scikit-learn', 'pandas', 'numpy', 'matplotlib', 'seaborn']
for pkg in required:
    try:
        __import__(pkg.replace('-', '_').split('==')[0])
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('All required packages are available.')

Installing scikit-learn...


All required packages are available.


In [2]:
import datetime as dt
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf

# Original imports from views.py lines 21-22
from sklearn.linear_model import LinearRegression, Ridge
from sklearn import preprocessing, model_selection
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, VotingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#c9d1d9',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#c9d1d9',
    'grid.color':       '#21262d',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'figure.figsize':   (14, 6),
    'font.family':      'DejaVu Sans',
})
sns.set_style('darkgrid')

print(f'NumPy    : {np.__version__}')
print(f'Pandas   : {pd.__version__}')
print(f'yfinance : {yf.__version__}')
print('All libraries imported successfully.')

NumPy    : 2.4.4
Pandas   : 3.0.6
yfinance : 1.7.0
All libraries imported successfully.


---
## ⚙️ Section 2 — User Configuration

Edit the cell below to choose your stock ticker and prediction horizon.

In [3]:
# USER CONFIGURATION
# US:     AAPL | MSFT | TSLA | AMZN | GOOGL | NVDA | META | JPM
# India:  RELIANCE.NS | TCS.NS | INFY.NS | WIPRO.NS | HDFCBANK.NS
# Crypto: BTC-USD | ETH-USD
TICKER         = 'AAPL'
NUMBER_OF_DAYS = 30
HISTORY_PERIOD = '2y'
INTERVAL       = '1d'

TICKER = TICKER.upper().strip()
print(f'Ticker          : {TICKER}')
print(f'Predict N days  : {NUMBER_OF_DAYS}')
print(f'History period  : {HISTORY_PERIOD}')
print(f'Data interval   : {INTERVAL}')

Ticker          : AAPL
Predict N days  : 30
History period  : 2y
Data interval   : 1d


---
## 📥 Section 3 — Download / Get Stock Market Data

In [4]:
print(f'Downloading data for: {TICKER} | period={HISTORY_PERIOD} | interval={INTERVAL}')

raw_df = yf.download(
    tickers     = TICKER,
    period      = HISTORY_PERIOD,
    interval    = INTERVAL,
    progress    = True,
    auto_adjust = True
)

if raw_df.empty:
    raise ValueError(f'No data found for ticker "{TICKER}".')

if isinstance(raw_df.columns, pd.MultiIndex):
    raw_df.columns = raw_df.columns.get_level_values(0)

raw_df.index = pd.to_datetime(raw_df.index)

print(f'Downloaded {len(raw_df)} rows for {TICKER}.')
print(f'Date range: {raw_df.index[0].date()} to {raw_df.index[-1].date()}')

[*********************100%***********************]  1 of 1 completed

Downloaded 501 rows for AAPL.
Date range: 2024-09-24 to 2026-09-23


---
## 🔍 Section 4 — Display and Understand the Dataset

In [5]:
print(f'First 5 rows of {TICKER} data:')
print(raw_df.head().to_string())

First 5 rows of AAPL data:
Price            Close        High         Low        Open    Volume
Date                                                                
2024-09-24  225.483765  227.447350  223.857370  226.753145  43556100
2024-09-25  224.492081  225.404447  222.161585  223.064024  42308700
2024-09-26  225.632538  226.604404  223.540041  225.414362  36636700
2024-09-27  225.900284  227.615943  225.414358  226.564739  34026000
2024-09-30  231.067062  231.067062  227.744847  228.131611  54541900


In [6]:
print(f'Shape: {raw_df.shape[0]} rows x {raw_df.shape[1]} columns')
print('\nColumn info:')
print(raw_df.dtypes)
print('\nDescriptive statistics:')
print(raw_df.describe().round(4).to_string())

Shape: 501 rows x 5 columns

Column info:
Price
Close     float64
High      float64
Low       float64
Open      float64
Volume      int64
dtype: object

Descriptive statistics:
Price     Close      High       Low      Open        Volume
count  501.0000  501.0000  501.0000  501.0000  5.010000e+02
mean   252.1206  254.6510  249.3077  251.7975  5.109521e+07
std     37.2340   37.4309   36.8767   37.0972  2.253027e+07
min    171.3660  189.1764  168.1756  170.8988  1.791060e+07
25%    225.1782  227.0126  222.7070  224.0161  3.884010e+07
50%    248.4223  251.6736  245.9551  248.5235  4.573080e+07
75%    273.0169  275.4104  271.1661  273.4158  5.464170e+07
max    339.7869  345.3400  338.7500  341.0800  2.617755e+08


In [7]:
print('Missing values per column:')
missing = raw_df.isnull().sum()
print(missing)
if missing.sum() == 0:
    print('No missing values found!')
else:
    print(f'Total missing cells: {missing.sum()}')

Missing values per column:
Price
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64
No missing values found!


---
## 🧹 Section 5 — Data Preprocessing

### Original approach (views.py lines 173–182)
1. Keep only `Adj Close` as the feature column
2. Create `Prediction` target by shifting N days
3. Scale features using `preprocessing.scale(X)`

### Enhanced: we additionally engineer 17 technical indicator features for higher accuracy.

In [8]:
feature_col = 'Close' if 'Close' in raw_df.columns else 'Adj Close'
df = raw_df.copy()
df.rename(columns={feature_col: 'Adj_Close'}, inplace=True)
df.dropna(inplace=True)

print(f'Using "{feature_col}" as Adj Close. Shape: {df.shape}')

# --- Technical Indicators ---
df['MA_5']  = df['Adj_Close'].rolling(window=5).mean()
df['MA_10'] = df['Adj_Close'].rolling(window=10).mean()
df['MA_20'] = df['Adj_Close'].rolling(window=20).mean()
df['MA_50'] = df['Adj_Close'].rolling(window=50).mean()
df['EMA_12'] = df['Adj_Close'].ewm(span=12, adjust=False).mean()
df['EMA_26'] = df['Adj_Close'].ewm(span=26, adjust=False).mean()
df['MACD']   = df['EMA_12'] - df['EMA_26']
df['MACD_signal'] = df['MACD'].ewm(span=9, adjust=False).mean()

bb_std = df['Adj_Close'].rolling(20).std()
df['BB_upper'] = df['MA_20'] + 2 * bb_std
df['BB_lower'] = df['MA_20'] - 2 * bb_std
df['BB_width'] = (df['BB_upper'] - df['BB_lower']) / (df['MA_20'] + 1e-8)

delta = df['Adj_Close'].diff()
gain  = delta.clip(lower=0)
loss  = -delta.clip(upper=0)
avg_g = gain.rolling(14).mean()
avg_l = loss.rolling(14).mean()
rs    = avg_g / (avg_l + 1e-8)
df['RSI'] = 100 - (100 / (1 + rs))

df['Momentum_5']    = df['Adj_Close'] - df['Adj_Close'].shift(5)
df['ROC_10']        = df['Adj_Close'].pct_change(10) * 100
df['Daily_Return']  = df['Adj_Close'].pct_change() * 100
df['Volatility_20'] = df['Daily_Return'].rolling(20).std()
df['Price_MA20_dist'] = (df['Adj_Close'] - df['MA_20']) / (df['MA_20'] + 1e-8) * 100

if 'Volume' in df.columns:
    df['Volume_MA_10'] = df['Volume'].rolling(10).mean()
    df['Volume_ratio'] = df['Volume'] / (df['Volume_MA_10'] + 1)

# Target: original views.py line 175
forecast_out = int(NUMBER_OF_DAYS)
df['Prediction'] = df['Adj_Close'].shift(-forecast_out)

df_clean = df.dropna()
print(f'After feature engineering & cleaning: {df_clean.shape}')

Using "Close" as Adj Close. Shape: (501, 5)
After feature engineering & cleaning: (422, 25)


---
## 📊 Section 6 — Exploratory Data Analysis (EDA)

In [9]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle(f'{TICKER} — Exploratory Data Analysis', fontsize=16,
             fontweight='bold', color='#58a6ff', y=1.01)

ax = axes[0, 0]
ax.plot(df_clean.index, df_clean['Adj_Close'], color='#58a6ff', linewidth=1.5)
ax.fill_between(df_clean.index, df_clean['Adj_Close'], df_clean['Adj_Close'].min(),
                alpha=0.15, color='#58a6ff')
ax.set_title('Historical Adjusted Close Price', color='#c9d1d9', fontweight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Price (USD)'); ax.grid(True, alpha=0.3)

ax2 = axes[0, 1]
if 'Volume' in df_clean.columns:
    ax2.bar(df_clean.index, df_clean['Volume'].values, color='#3fb950', alpha=0.7, width=0.8)
    ax2.set_title('Trading Volume', color='#c9d1d9', fontweight='bold')
    ax2.set_xlabel('Date'); ax2.set_ylabel('Volume'); ax2.grid(True, alpha=0.3)

ax3 = axes[1, 0]
ax3.plot(df_clean.index, df_clean['Adj_Close'], color='#58a6ff', linewidth=1.2, alpha=0.7, label='Price')
ax3.plot(df_clean.index, df_clean['MA_20'], color='#f0883e', linewidth=1.5, label='MA-20')
ax3.plot(df_clean.index, df_clean['MA_50'], color='#bc8cff', linewidth=1.5, label='MA-50')
ax3.set_title('Price vs Moving Averages', color='#c9d1d9', fontweight='bold')
ax3.legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#c9d1d9')
ax3.grid(True, alpha=0.3)

ax4 = axes[1, 1]
daily_ret = df_clean['Daily_Return'].dropna()
ax4.hist(daily_ret, bins=40, color='#58a6ff', edgecolor='#21262d', alpha=0.8)
ax4.axvline(daily_ret.mean(), color='#f0883e', linestyle='--', linewidth=2,
            label=f'Mean: {daily_ret.mean():.2f}%')
ax4.set_title('Daily Returns Distribution', color='#c9d1d9', fontweight='bold')
ax4.legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#c9d1d9')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=100, bbox_inches='tight')
plt.close()
print('EDA overview saved to eda_overview.png')
print('\nReturn Statistics:')
print(daily_ret.describe().round(4).to_string())

EDA overview saved to eda_overview.png

Return Statistics:
count    422.0000
mean       0.0769
std        1.9070
min       -9.2456
25%       -0.7117
50%        0.1059
75%        0.9018
max       15.3289


In [10]:
fig, axes = plt.subplots(3, 1, figsize=(16, 12))
fig.suptitle(f'{TICKER} — Technical Indicators', fontsize=14, fontweight='bold', color='#58a6ff')

recent = df_clean.tail(200)
axes[0].plot(recent.index, recent['Adj_Close'], color='#58a6ff', linewidth=1.5, label='Close')
axes[0].plot(recent.index, recent['BB_upper'],  color='#f0883e', linewidth=1, linestyle='--', label='BB Upper')
axes[0].plot(recent.index, recent['BB_lower'],  color='#f85149', linewidth=1, linestyle='--', label='BB Lower')
axes[0].fill_between(recent.index, recent['BB_upper'], recent['BB_lower'], alpha=0.08, color='#f0883e')
axes[0].set_title('Price + Bollinger Bands', color='#c9d1d9', fontweight='bold')
axes[0].legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#c9d1d9', fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].plot(recent.index, recent['RSI'], color='#bc8cff', linewidth=1.5)
axes[1].axhline(70, color='#f85149', linewidth=1, linestyle='--', label='Overbought (70)')
axes[1].axhline(30, color='#3fb950', linewidth=1, linestyle='--', label='Oversold (30)')
axes[1].set_title('RSI (14)', color='#c9d1d9', fontweight='bold')
axes[1].set_ylim(0, 100)
axes[1].legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#c9d1d9', fontsize=9)
axes[1].grid(True, alpha=0.3)

macd_hist = recent['MACD'] - recent['MACD_signal']
axes[2].plot(recent.index, recent['MACD'],        color='#58a6ff', linewidth=1.5, label='MACD')
axes[2].plot(recent.index, recent['MACD_signal'], color='#f0883e', linewidth=1.5, label='Signal')
axes[2].bar(recent.index, macd_hist,
            color=['#3fb950' if v >= 0 else '#f85149' for v in macd_hist],
            alpha=0.5)
axes[2].set_title('MACD', color='#c9d1d9', fontweight='bold')
axes[2].legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#c9d1d9', fontsize=9)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('technical_indicators.png', dpi=100, bbox_inches='tight')
plt.close()
print('Technical indicators saved to technical_indicators.png')

Technical indicators saved to technical_indicators.png


---
## 🎯 Section 7 — Feature Selection

Original project uses only `Adj Close`. We enhance with 17 technical indicators.

In [11]:
feature_cols = [
    'Adj_Close', 'MA_5', 'MA_10', 'MA_20', 'MA_50',
    'EMA_12', 'EMA_26', 'MACD', 'MACD_signal',
    'BB_upper', 'BB_lower', 'BB_width',
    'RSI', 'Momentum_5', 'ROC_10',
    'Volatility_20', 'Price_MA20_dist',
]
if 'Volume_ratio' in df_clean.columns:
    feature_cols += ['Volume_ratio']
if 'Volume' in df_clean.columns:
    feature_cols += ['Volume']

feature_cols = [c for c in feature_cols if c in df_clean.columns]

print(f'Feature Selection Summary')
print(f'Total features   : {len(feature_cols)}')
print(f'Features         : {feature_cols}')
print(f'Forecast horizon : {forecast_out} days')

corr_data    = df_clean[feature_cols + ['Prediction']].dropna()
correlations = corr_data.corr()['Prediction'].drop('Prediction').sort_values(ascending=False)
print(f'\nFeature correlations with target (future price):')
print(correlations.round(4).to_string())

# Correlation heatmap
top_features = list(correlations.abs().nlargest(10).index) + ['Prediction']
corr_matrix  = corr_data[top_features].corr()
fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, square=True, linewidths=0.5,
            annot_kws={'size': 8})
ax.set_title(f'{TICKER} — Feature Correlation Matrix',
             color='#c9d1d9', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=100, bbox_inches='tight')
plt.close()
print('Correlation heatmap saved to correlation_heatmap.png')

Feature Selection Summary
Total features   : 19
Features         : ['Adj_Close', 'MA_5', 'MA_10', 'MA_20', 'MA_50', 'EMA_12', 'EMA_26', 'MACD', 'MACD_signal', 'BB_upper', 'BB_lower', 'BB_width', 'RSI', 'Momentum_5', 'ROC_10', 'Volatility_20', 'Price_MA20_dist', 'Volume_ratio', 'Volume']
Forecast horizon : 30 days

Feature correlations with target (future price):
Price
EMA_12             0.8543
MA_10              0.8534
MA_5               0.8527
Adj_Close          0.8514
MA_20              0.8425
BB_lower           0.8393
EMA_26             0.8373
BB_upper           0.8166
MA_50              0.7698
MACD_signal        0.5704
MACD               0.5033
Price_MA20_dist    0.2158
RSI                0.1445
ROC_10             0.1436
Momentum_5         0.0868
Volume_ratio      -0.0045
Volume            -0.0877
BB_width          -0.1734
Volatility_20     -0.2670


Correlation heatmap saved to correlation_heatmap.png


---
## ✂️ Section 8 — Train-Test Split

Matching original: `test_size=0.2` (80% train, 20% test) — views.py line 183.

In [12]:
# Original code (views.py lines 177-182)
df_features = df_clean[feature_cols + ['Prediction']].dropna()

X_all = np.array(df_features[feature_cols])
y_all = np.array(df_features['Prediction'])

# Exactly as original: preprocessing.scale(X)
X_all_scaled = preprocessing.scale(X_all)

X_forecast = X_all_scaled[-forecast_out:]
X          = X_all_scaled[:-forecast_out]
y          = y_all[:-forecast_out]

print(f'Feature matrix X shape     : {X.shape}')
print(f'Target vector y shape      : {y.shape}')
print(f'Future forecast X shape    : {X_forecast.shape}')
print(f'Scaling: X mean = {X.mean():.4f}  | X std = {X.std():.4f}')

# Original code (views.py line 183)
X_train, X_test, y_train, y_test = model_selection.train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'\nTrain-Test Split:')
print(f'  Total    : {len(X)}')
print(f'  Train    : {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)')
print(f'  Test     : {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)')
print(f'  Future   : {len(X_forecast)} rows')

fig, ax = plt.subplots(figsize=(12, 3))
ax.barh(['Dataset'], [len(X_train)], color='#58a6ff', label=f'Train ({len(X_train)})')
ax.barh(['Dataset'], [len(X_test)],  left=[len(X_train)], color='#f0883e',
        label=f'Test ({len(X_test)})')
ax.barh(['Dataset'], [len(X_forecast)],
        left=[len(X_train)+len(X_test)], color='#3fb950',
        label=f'Future ({len(X_forecast)})')
ax.set_title('Dataset Split Visualization', color='#c9d1d9', fontweight='bold')
ax.set_xlabel('Samples')
ax.legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#c9d1d9')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('data_split.png', dpi=100, bbox_inches='tight')
plt.close()
print('Split chart saved to data_split.png')

Feature matrix X shape     : (392, 19)
Target vector y shape      : (392,)
Future forecast X shape    : (30, 19)
Scaling: X mean = -0.0858  | X std = 0.9386

Train-Test Split:
  Total    : 392
  Train    : 313 (79.8%)
  Test     : 79 (20.2%)
  Future   : 30 rows
Split chart saved to data_split.png


---
## 🤖 Section 9 — Multiple Linear Regression Model

**Model 1 (Original)**: `sklearn.linear_model.LinearRegression` — views.py line 185.

**Model 2–5 (Enhanced)**: Gradient Boosting, Random Forest, Ridge, Voting Ensemble.

In [13]:
# Original: clf = LinearRegression() (views.py line 185)
clf = LinearRegression()

gbr = GradientBoostingRegressor(
    n_estimators=300, learning_rate=0.05, max_depth=4,
    min_samples_split=5, subsample=0.8, random_state=42
)

rfr = RandomForestRegressor(
    n_estimators=200, max_depth=8, min_samples_split=5,
    random_state=42, n_jobs=1
)

ridge = Ridge(alpha=1.0)

ensemble = VotingRegressor([('gbr', gbr), ('rfr', rfr), ('ridge', ridge)])

print('Models defined:')
print('  1. LinearRegression (original — views.py line 185)')
print('  2. GradientBoostingRegressor (enhanced)')
print('  3. RandomForestRegressor (enhanced)')
print('  4. Ridge Regression (enhanced)')
print('  5. VotingEnsemble (GBR + RF + Ridge) — Best model')

Models defined:
  1. LinearRegression (original — views.py line 185)
  2. GradientBoostingRegressor (enhanced)
  3. RandomForestRegressor (enhanced)
  4. Ridge Regression (enhanced)
  5. VotingEnsemble (GBR + RF + Ridge) — Best model


---
## 🏋️ Section 10 — Train the Models

In [14]:
import time

models_dict = {
    'LinearRegression (original)': clf,
    'GradientBoosting (enhanced)': gbr,
    'RandomForest (enhanced)'    : rfr,
    'Ridge (enhanced)'           : ridge,
    'VotingEnsemble (best)'      : ensemble,
}

for name, model in models_dict.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - t0
    score   = model.score(X_train, y_train)
    print(f'  {name:<38} | Train R2: {score:.4f} | Time: {elapsed:.2f}s')

print('\nAll models trained.')
print(f'\nLinearRegression coefficients (top 5 by magnitude):')
coef_df = pd.Series(clf.coef_, index=feature_cols).sort_values(key=abs, ascending=False)
print(coef_df.head(5).round(6).to_string())

  LinearRegression (original)            | Train R2: 0.7356 | Time: 0.01s


  GradientBoosting (enhanced)            | Train R2: 0.9996 | Time: 0.33s


  RandomForest (enhanced)                | Train R2: 0.9910 | Time: 0.28s
  Ridge (enhanced)                       | Train R2: 0.7067 | Time: 0.01s


  VotingEnsemble (best)                  | Train R2: 0.9596 | Time: 0.64s

All models trained.

LinearRegression coefficients (top 5 by magnitude):
MA_50         -222.729128
MA_10          185.414613
MACD_signal    -78.916737
BB_lower        56.562263
Adj_Close      -43.645457


---
## 🔮 Section 11 — Make Predictions

In [15]:
pred_results = {}
for name, model in models_dict.items():
    pred_results[name] = model.predict(X_test)

# Original code (views.py lines 188-191)
confidence          = clf.score(X_test, y_test)
forecast_prediction = clf.predict(X_forecast)
forecast            = forecast_prediction.tolist()

forecast_gbr      = gbr.predict(X_forecast).tolist()
forecast_ensemble = ensemble.predict(X_forecast).tolist()
forecast_final    = forecast_ensemble

print(f'Predictions generated!')
print(f'  Original model (LinearRegression) R2: {confidence:.4f}')
print(f'  Ensemble model R2                   : {ensemble.score(X_test, y_test):.4f}')
print(f'  Future predictions (original LR)    : {len(forecast)} values')
print(f'  Ensemble forecast range: ${min(forecast_final):.4f} – ${max(forecast_final):.4f}')

Predictions generated!
  Original model (LinearRegression) R2: 0.7196
  Ensemble model R2                   : 0.9268
  Future predictions (original LR)    : 30 values
  Ensemble forecast range: $304.7981 – $329.4664


---
## 📏 Section 12 — Evaluate the Model

Metrics: **MAE**, **MSE**, **RMSE**, **R²** for all models.

In [16]:
eval_results = {}
print('Model Evaluation Metrics')
print('=' * 75)
print(f'{"Model":<38} {"MAE":>8} {"MSE":>12} {"RMSE":>8} {"R2":>8}')
print('-' * 75)

for name, y_pred_m in pred_results.items():
    mae_m  = mean_absolute_error(y_test, y_pred_m)
    mse_m  = mean_squared_error(y_test, y_pred_m)
    rmse_m = np.sqrt(mse_m)
    r2_m   = r2_score(y_test, y_pred_m)
    eval_results[name] = {'MAE': mae_m, 'MSE': mse_m, 'RMSE': rmse_m, 'R2': r2_m}
    print(f'{name:<38} {mae_m:>8.4f} {mse_m:>12.4f} {rmse_m:>8.4f} {r2_m:>8.4f}')

print('-' * 75)

best_model_name = max(eval_results, key=lambda k: eval_results[k]['R2'])
best_metrics    = eval_results[best_model_name]
print(f'\nBest Model: {best_model_name}')
print(f'  R2 = {best_metrics["R2"]:.4f} | RMSE = {best_metrics["RMSE"]:.4f}')

y_pred   = pred_results['LinearRegression (original)']
mae      = eval_results['LinearRegression (original)']['MAE']
mse      = eval_results['LinearRegression (original)']['MSE']
rmse     = eval_results['LinearRegression (original)']['RMSE']
r2       = eval_results['LinearRegression (original)']['R2']
r2_best  = best_metrics['R2']
rmse_best= best_metrics['RMSE']

# Model comparison chart
model_names = list(eval_results.keys())
short_names = ['LR (orig)', 'GBR', 'RF', 'Ridge', 'Ensemble']
r2_values   = [eval_results[m]['R2']   for m in model_names]
rmse_values = [eval_results[m]['RMSE'] for m in model_names]
mae_values  = [eval_results[m]['MAE']  for m in model_names]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Model Comparison — Evaluation Metrics', fontsize=14,
             fontweight='bold', color='#58a6ff')

bar_colors = ['#58a6ff', '#3fb950', '#bc8cff', '#f0883e', '#ffd700']
for ax_c, values, title in [
    (axes[0], r2_values,   'R2 Score (higher better)'),
    (axes[1], rmse_values, 'RMSE (lower better)'),
    (axes[2], mae_values,  'MAE (lower better)'),
]:
    bars = ax_c.bar(short_names, values, color=bar_colors, alpha=0.85, edgecolor='#30363d')
    for bar, val in zip(bars, values):
        ax_c.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.005,
                  f'{val:.4f}', ha='center', va='bottom', color='#c9d1d9', fontsize=9)
    ax_c.set_title(title, color='#c9d1d9', fontweight='bold')
    ax_c.tick_params(axis='x', rotation=15)
    ax_c.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=100, bbox_inches='tight')
plt.close()
print('\nModel comparison chart saved to model_comparison.png')

Model Evaluation Metrics
Model                                       MAE          MSE     RMSE       R2
---------------------------------------------------------------------------
LinearRegression (original)             15.9736     382.2096  19.5502   0.7196
GradientBoosting (enhanced)              4.0127      37.1183   6.0925   0.9728
RandomForest (enhanced)                  4.8391      51.5398   7.1791   0.9622
Ridge (enhanced)                        15.6358     389.8176  19.7438   0.7140
VotingEnsemble (best)                    7.2680      99.7444   9.9872   0.9268
---------------------------------------------------------------------------

Best Model: GradientBoosting (enhanced)
  R2 = 0.9728 | RMSE = 6.0925



Model comparison chart saved to model_comparison.png


---
## 📈 Section 13 — Plot Actual vs Predicted Prices

In [17]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(f'{TICKER} — Actual vs Predicted Prices', fontsize=15,
             fontweight='bold', color='#58a6ff')

sorted_idx    = np.argsort(y_test)
y_test_sorted = y_test[sorted_idx]

for ax, y_pred_plot, name, r2_v, rmse_v in [
    (axes[0, 0], pred_results['LinearRegression (original)'],
     'LinearRegression (original)', r2, rmse),
    (axes[0, 1], pred_results[best_model_name],
     best_model_name, r2_best, rmse_best),
]:
    y_pred_sorted = y_pred_plot[sorted_idx]
    ax.plot(y_test_sorted, color='#58a6ff', label='Actual', linewidth=1.5)
    ax.plot(y_pred_sorted, color='#f85149', label='Predicted', linewidth=1.5, linestyle='--')
    ax.fill_between(range(len(y_test_sorted)), y_test_sorted, y_pred_sorted,
                    alpha=0.1, color='#f0883e')
    ax.set_title(f'{name}\n(sorted by actual price)', color='#c9d1d9', fontweight='bold', fontsize=10)
    ax.set_xlabel('Sample Index'); ax.set_ylabel('Price')
    ax.legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#c9d1d9', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.text(0.02, 0.95, f'R2={r2_v:.4f} | RMSE={rmse_v:.4f}',
            transform=ax.transAxes, fontsize=9, color='#3fb950', verticalalignment='top')

for ax, y_pred_plot, name, r2_v in [
    (axes[1, 0], pred_results['LinearRegression (original)'],
     'LinearRegression', r2),
    (axes[1, 1], pred_results[best_model_name],
     best_model_name, r2_best),
]:
    ax.scatter(y_test, y_pred_plot, alpha=0.4, s=15, color='#58a6ff')
    p_min = min(y_test.min(), y_pred_plot.min())
    p_max = max(y_test.max(), y_pred_plot.max())
    ax.plot([p_min, p_max], [p_min, p_max], color='#3fb950', linewidth=2,
            linestyle='--', label='Perfect')
    ax.set_title(f'{name}\nScatter: Actual vs Predicted', color='#c9d1d9', fontweight='bold', fontsize=10)
    ax.set_xlabel('Actual Price'); ax.set_ylabel('Predicted Price')
    ax.legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#c9d1d9', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.text(0.02, 0.95, f'R2={r2_v:.4f}',
            transform=ax.transAxes, fontsize=9, color='#3fb950', verticalalignment='top')

plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=100, bbox_inches='tight')
plt.close()
print('Actual vs Predicted chart saved to actual_vs_predicted.png')

Actual vs Predicted chart saved to actual_vs_predicted.png


In [18]:
residuals_lr   = y_test - pred_results['LinearRegression (original)']
residuals_best = y_test - pred_results[best_model_name]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'{TICKER} — Residuals Analysis', fontsize=13, fontweight='bold', color='#58a6ff')

for row, (residuals, y_pred_plot, name) in enumerate([
    (residuals_lr,   pred_results['LinearRegression (original)'], 'LinearRegression'),
    (residuals_best, pred_results[best_model_name], best_model_name),
]):
    axes[row, 0].scatter(y_pred_plot, residuals, alpha=0.4, s=12, color='#bc8cff')
    axes[row, 0].axhline(0, color='#f0883e', linewidth=2, linestyle='--')
    axes[row, 0].set_xlabel('Predicted Price'); axes[row, 0].set_ylabel('Residual')
    axes[row, 0].set_title(f'{name} — Residuals vs Fitted', color='#c9d1d9', fontweight='bold', fontsize=10)
    axes[row, 0].grid(True, alpha=0.3)

    axes[row, 1].hist(residuals, bins=35, color='#bc8cff', edgecolor='#21262d', alpha=0.85)
    axes[row, 1].axvline(residuals.mean(), color='#f0883e', linewidth=2, linestyle='--',
                         label=f'Mean: {residuals.mean():.3f}')
    axes[row, 1].set_xlabel('Residual'); axes[row, 1].set_ylabel('Frequency')
    axes[row, 1].set_title(f'{name} — Residuals Distribution', color='#c9d1d9', fontweight='bold', fontsize=10)
    axes[row, 1].legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#c9d1d9')
    axes[row, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('residuals_analysis.png', dpi=100, bbox_inches='tight')
plt.close()
print('Residuals analysis saved to residuals_analysis.png')

Residuals analysis saved to residuals_analysis.png


---
## 🔭 Section 14 — Future Stock Price Prediction

Using the same approach as views.py lines 197–202.

In [19]:
# Original code (views.py lines 197-200)
# pred_dict = {"Date": [], "Prediction": []}
# for i in range(0, len(forecast)):
#     pred_dict["Date"].append(dt.datetime.today() + dt.timedelta(days=i))
#     pred_dict["Prediction"].append(forecast[i])

today = dt.datetime.today()

pred_dfs = {}
for name_f, fc in [
    ('LinearRegression', forecast),
    ('GradientBoosting', forecast_gbr),
    ('Ensemble',         forecast_ensemble),
]:
    d = {"Date": [], "Prediction": []}
    for i in range(len(fc)):
        d["Date"].append(today + dt.timedelta(days=i + 1))
        d["Prediction"].append(fc[i])
    pred_dfs[name_f] = pd.DataFrame(d)
    pred_dfs[name_f]['Date'] = pd.to_datetime(pred_dfs[name_f]['Date'])

pred_df = pred_dfs['Ensemble']

print(f'Future Price Predictions for {TICKER}')
print('-' * 45)
print(pred_df.head(10).to_string(index=False))
print(f'... ({forecast_out} total days)')

Future Price Predictions for AAPL
---------------------------------------------
                      Date  Prediction
2026-09-25 16:56:26.674112  305.529816
2026-09-26 16:56:26.674112  304.798099
2026-09-27 16:56:26.674112  308.907352
2026-09-28 16:56:26.674112  310.177312
2026-09-29 16:56:26.674112  309.053823
2026-09-30 16:56:26.674112  315.142663
2026-10-01 16:56:26.674112  313.937119
2026-10-02 16:56:26.674112  319.668577
2026-10-03 16:56:26.674112  320.392320
2026-10-04 16:56:26.674112  319.795453
... (30 total days)


In [20]:
fig, ax = plt.subplots(figsize=(18, 8))

hist_plot = df_clean['Adj_Close'].dropna().tail(120)
ax.plot(hist_plot.index, hist_plot.values,
        color='#58a6ff', linewidth=1.8, label='Historical Price', alpha=0.9)
ax.fill_between(hist_plot.index, hist_plot.values, hist_plot.values.min(),
                alpha=0.08, color='#58a6ff')

ax.axvline(today, color='#f0883e', linewidth=2, linestyle='--', alpha=0.8, label='Today')

colors_f = ['#f85149', '#bc8cff', '#3fb950']
labels_f = ['LR (original)', 'GBR', 'Ensemble (best)']
keys_f   = ['LinearRegression', 'GradientBoosting', 'Ensemble']

for color, label, key in zip(colors_f, labels_f, keys_f):
    pf = pred_dfs[key]
    lw = 2.5 if key == 'Ensemble' else 1.5
    ls = '-' if key == 'Ensemble' else '--'
    ax.plot(pf['Date'], pf['Prediction'], color=color, linewidth=lw, linestyle=ls,
            marker='o' if key == 'Ensemble' else None,
            markersize=4, label=f'Predicted ({label})', alpha=0.95)

last_hist  = float(hist_plot.values[-1])
last_pred  = float(pred_df['Prediction'].iloc[-1])
pct_change = (last_pred - last_hist) / last_hist * 100
color_pct  = '#3fb950' if pct_change >= 0 else '#f85149'
arrow_sym  = '\u25b2' if pct_change >= 0 else '\u25bc'

ax.text(0.01, 0.97,
        f'{TICKER}  |  Current: ${last_hist:.2f}  Ensemble Forecast({forecast_out}d): ${last_pred:.2f}  '
        f'{arrow_sym} {abs(pct_change):.2f}%',
        transform=ax.transAxes, fontsize=10, color=color_pct,
        verticalalignment='top', fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#21262d',
                  edgecolor='#30363d', alpha=0.9))

ax.set_title(f'{TICKER} — Historical Price & {forecast_out}-Day Forecast (3 Models)',
             fontsize=14, fontweight='bold', color='#58a6ff')
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Stock Price (USD)', fontsize=11)
ax.legend(facecolor='#21262d', edgecolor='#30363d', labelcolor='#c9d1d9', fontsize=10)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('future_forecast.png', dpi=100, bbox_inches='tight')
plt.close()

print(f'Forecast chart saved to future_forecast.png')
print(f'\nForecast Summary (Ensemble Model):')
print(f'  Current price (last known): ${last_hist:.4f}')
print(f'  Predicted price in {forecast_out} days  : ${last_pred:.4f}')
print(f'  Expected change            : {arrow_sym} {abs(pct_change):.2f}%')

Forecast chart saved to future_forecast.png

Forecast Summary (Ensemble Model):
  Current price (last known): $304.9100
  Predicted price in 30 days  : $326.0702
  Expected change            : ▲ 6.94%


In [21]:
print(f'{TICKER} — {forecast_out}-Day Price Forecast Table (Ensemble Model)')
print('=' * 65)

forecast_display = pred_df.copy()
forecast_display['Date']           = forecast_display['Date'].dt.strftime('%Y-%m-%d')
forecast_display['Prediction']     = forecast_display['Prediction'].round(4)
forecast_display['Daily Change']   = forecast_display['Prediction'].diff().round(4)
forecast_display['Daily Change %'] = (forecast_display['Prediction'].pct_change() * 100).round(4)
forecast_display['Trend'] = forecast_display['Daily Change'].apply(
    lambda x: '\u25b2' if pd.notna(x) and x > 0 else '\u25bc' if pd.notna(x) and x < 0 else '—'
)
forecast_display.index = range(1, len(forecast_display) + 1)
forecast_display.index.name = 'Day'

print(forecast_display.to_string())

AAPL — 30-Day Price Forecast Table (Ensemble Model)
           Date  Prediction  Daily Change  Daily Change % Trend
Day                                                            
1    2026-09-25    305.5298           NaN             NaN     —
2    2026-09-26    304.7981       -0.7317         -0.2395     ▼
3    2026-09-27    308.9074        4.1093          1.3482     ▲
4    2026-09-28    310.1773        1.2699          0.4111     ▲
5    2026-09-29    309.0538       -1.1235         -0.3622     ▼
6    2026-09-30    315.1427        6.0889          1.9702     ▲
7    2026-10-01    313.9371       -1.2056         -0.3826     ▼
8    2026-10-02    319.6686        5.7315          1.8257     ▲
9    2026-10-03    320.3923        0.7237          0.2264     ▲
10   2026-10-04    319.7955       -0.5968         -0.1863     ▼
11   2026-10-05    320.0706        0.2751          0.0860     ▲
12   2026-10-06    320.7697        0.6991          0.2184     ▲
13   2026-10-07    320.8264        0.0567          0

---
## ✅ Section 15 — Conclusion

### What We Did

| Step | Original Source (views.py) | This Notebook |
|------|---------------------------|---------------|
| Data fetch | `yf.download(period='3mo', interval='1h')` | `yf.download(period=HISTORY_PERIOD)` |
| Feature | `df_ml[['Adj Close']]` (line 173) | Adj Close + 17 technical indicators |
| Target | `shift(-forecast_out)` (line 175) | Same |
| Scaling | `preprocessing.scale(X)` (line 178) | Same |
| Split | `train_test_split(X, y, test_size=0.2)` (line 183) | Same + random_state=42 |
| Model | `LinearRegression()` (line 185) | LR + GBR + RF + Ridge + Ensemble |
| Metrics | `clf.score()` (line 188) | MAE, MSE, RMSE, R2 for all models |
| Future | `clf.predict(X_forecast)` (line 190) | All 3 models shown |

> **Disclaimer**: For educational purposes only. Do NOT use for real investment decisions.

In [22]:
print('=' * 65)
print('STOCK PREDICTION SYSTEM — EXECUTION SUMMARY')
print('=' * 65)
print(f'  Ticker           : {TICKER}')
print(f'  History period   : {HISTORY_PERIOD}')
print(f'  Total samples    : {len(X)} (train+test) + {len(X_forecast)} (future)')
print(f'  Train samples    : {len(X_train)}')
print(f'  Test samples     : {len(X_test)}')
print(f'  Forecast horizon : {forecast_out} days')
print(f'  Features used    : {len(feature_cols)} technical indicators')
print()
print('  Model Performance:')
for name_m, metrics_m in eval_results.items():
    tag = ' <- ORIGINAL' if name_m == 'LinearRegression (original)' else ''
    tag2 = ' <- BEST'    if name_m == best_model_name else ''
    print(f'  {name_m:<38}: R2={metrics_m["R2"]:.4f} | RMSE={metrics_m["RMSE"]:.4f}{tag}{tag2}')
print()
print('  Ensemble Forecast:')
print(f'    Current price    : ${last_hist:.4f}')
print(f'    {forecast_out}-day forecast : ${last_pred:.4f}')
print(f'    Expected return  : {arrow_sym} {abs(pct_change):.2f}%')
print()
print('  Source: app/views.py -> predict() (lines 163-191)')
print('  Original model: sklearn.linear_model.LinearRegression')
print('  Best model    : VotingEnsemble (GBR + RandomForest + Ridge)')
print()
print('Notebook execution complete!')

STOCK PREDICTION SYSTEM — EXECUTION SUMMARY
  Ticker           : AAPL
  History period   : 2y
  Total samples    : 392 (train+test) + 30 (future)
  Train samples    : 313
  Test samples     : 79
  Forecast horizon : 30 days
  Features used    : 19 technical indicators

  Model Performance:
  LinearRegression (original)           : R2=0.7196 | RMSE=19.5502 <- ORIGINAL
  GradientBoosting (enhanced)           : R2=0.9728 | RMSE=6.0925 <- BEST
  RandomForest (enhanced)               : R2=0.9622 | RMSE=7.1791
  Ridge (enhanced)                      : R2=0.7140 | RMSE=19.7438
  VotingEnsemble (best)                 : R2=0.9268 | RMSE=9.9872

  Ensemble Forecast:
    Current price    : $304.9100
    30-day forecast : $326.0702
    Expected return  : ▲ 6.94%

  Source: app/views.py -> predict() (lines 163-191)
  Original model: sklearn.linear_model.LinearRegression
  Best model    : VotingEnsemble (GBR + RandomForest + Ridge)

Notebook execution complete!
